# Python Tricky Questions - Interview Gotchas

## Purpose of This Notebook

This notebook covers **common Python "trap" questions** that interviewers love to ask.
Understanding these will help you:
- Avoid subtle bugs in production code
- Demonstrate deep Python knowledge in interviews
- Debug issues faster

| Topic | The Trap | Correct Approach |
|-------|----------|------------------|
| `is` vs `==` | Using `is` for value comparison | Use `==` for values, `is` only for None |
| Mutable defaults | `def f(items=[])` shares list! | Use `items=None` |
| List copying | `b = a` creates reference | Use `b = a.copy()` |
| Late binding | Lambda captures variable, not value | Use default arg `lambda x=i: x` |
| Truthiness | `0` and `""` are falsy! | Explicit checks when needed |

---

## 1. `is` vs `==` - Identity vs Equality

### The Difference
- **`==`** checks **VALUE**: Are the contents the same?
- **`is`** checks **IDENTITY**: Are they the **SAME OBJECT** in memory?

### The Rule
- Use `==` for comparing values (99% of the time)
- Use `is` ONLY for: `None`, `True`, `False`

In [ ]:
# EXAMPLE 1: Small integers are cached by Python

# Python caches integers from -5 to 256 for performance
a = 256
b = 256

print("Small integer (256):")
print(f"  a == b: {a == b}")  # True - same VALUE
print(f"  a is b: {a is b}")  # True - SAME OBJECT (cached!)
print(f"  id(a) = {id(a)}, id(b) = {id(b)}")

# TRAP: Integers > 256 are NOT cached!
a = 257
b = 257

print("\nLarger integer (257):")
print(f"  a == b: {a == b}")  # True - same VALUE
print(f"  a is b: {a is b}")  # FALSE! Different objects!
print(f"  id(a) = {id(a)}, id(b) = {id(b)}")

print("\n⚠️ LESSON: Never use 'is' to compare integers!")

In [ ]:
# EXAMPLE 2: String interning

# Python "interns" (caches) short simple strings
s1 = "hello"
s2 = "hello"
s3 = "hel" + "lo"  # Optimized at compile time
s4 = "".join(["h", "e", "l", "l", "o"])  # Created at runtime

print("String interning:")
print(f"  s1 is s2: {s1 is s2}")  # True - both interned
print(f"  s1 is s3: {s1 is s3}")  # True - compile-time optimization
print(f"  s1 is s4: {s1 is s4}")  # FALSE! Runtime string not interned
print(f"  s1 == s4: {s1 == s4}")  # True - same VALUE

print("\n⚠️ LESSON: Never use 'is' to compare strings!")

In [ ]:
# CORRECT way to check for None

x = None

# ✅ CORRECT - use 'is' for None
if x is None:
    print("✅ x is None (correct way)")

# ❌ BAD PRACTICE - works but wrong idiom
if x == None:
    print("❌ x == None (bad practice)")

# WHY 'is' for None?
# - None is a singleton (only one None object exists)
# - 'is' is faster than '=='
# - Some objects override __eq__ in weird ways

---
## 2. Mutable Default Arguments (CLASSIC TRAP!)

### The Problem
```python
def add_item(item, items=[]):  # ❌ DON'T DO THIS!
    items.append(item)
    return items
```

### Why It's Wrong
- Default `[]` is created **ONCE** when function is defined
- **NOT** each time function is called!
- All calls share the SAME list!

In [ ]:
# ❌ BAD CODE - mutable default argument

def add_item_bad(item, items=[]):
    """
    THIS IS BUGGY!
    The default [] is created ONCE and shared between calls.
    """
    items.append(item)
    return items

print("❌ Using mutable default argument:")
print(f"   Call 1: add_item_bad('a') = {add_item_bad('a')}")  # ['a']
print(f"   Call 2: add_item_bad('b') = {add_item_bad('b')}")  # ['a', 'b'] - WRONG!
print(f"   Call 3: add_item_bad('c') = {add_item_bad('c')}")  # ['a', 'b', 'c'] - WRONG!

print("\n   The same list is being reused and modified each time!")

In [ ]:
# ✅ CORRECT CODE - use None as default

def add_item_good(item, items=None):
    """
    CORRECT pattern:
    1. Use None as default (None is immutable)
    2. Create new list inside function if None
    """
    if items is None:
        items = []  # Fresh list for each call
    items.append(item)
    return items

print("✅ Using None as default:")
print(f"   Call 1: add_item_good('a') = {add_item_good('a')}")  # ['a']
print(f"   Call 2: add_item_good('b') = {add_item_good('b')}")  # ['b'] - CORRECT!
print(f"   Call 3: add_item_good('c') = {add_item_good('c')}")  # ['c'] - CORRECT!

# Can still pass explicit list
existing = ['x', 'y']
print(f"\n   With existing list: {add_item_good('z', existing)}")

---
## 3. List Assignment vs Copy

### The Problem
```python
a = [1, 2, 3]
b = a  # ❌ This is NOT a copy!
b.append(4)
print(a)  # [1, 2, 3, 4] - a is also modified!
```

In [ ]:
# ❌ Assignment creates a REFERENCE, not a copy

original = [1, 2, 3]
reference = original  # Both point to SAME list in memory

print("Before modification:")
print(f"  original:  {original}  (id: {id(original)})")
print(f"  reference: {reference}  (id: {id(reference)})")
print(f"  Same object? {original is reference}")

reference.append(4)  # Modifying reference

print("\nAfter modifying reference:")
print(f"  original:  {original}  ← ALSO CHANGED!")
print(f"  reference: {reference}")

In [ ]:
# ✅ CORRECT: Create a copy

original = [1, 2, 3]

# Three ways to copy:
copy1 = original.copy()     # Method 1: .copy()
copy2 = list(original)      # Method 2: list()
copy3 = original[:]         # Method 3: slice

print("Creating copies:")
print(f"  original: {original}  (id: {id(original)})")
print(f"  copy1:    {copy1}  (id: {id(copy1)})")
print(f"  copy2:    {copy2}  (id: {id(copy2)})")
print(f"  copy3:    {copy3}  (id: {id(copy3)})")

copy1.append(99)

print("\nAfter modifying copy1:")
print(f"  original: {original}  ← UNCHANGED!")
print(f"  copy1:    {copy1}")

In [ ]:
# ⚠️ SHALLOW vs DEEP copy for nested structures

import copy

nested = [[1, 2], [3, 4]]

shallow = nested.copy()      # Shallow: copies outer list only
deep = copy.deepcopy(nested) # Deep: copies everything recursively

print("Nested structure:")
print(f"  original: {nested}")
print(f"  shallow:  {shallow}")
print(f"  deep:     {deep}")

# Modify inner list
nested[0][0] = 999

print("\nAfter nested[0][0] = 999:")
print(f"  original: {nested}")
print(f"  shallow:  {shallow}  ← ALSO MODIFIED! (shares inner lists)")
print(f"  deep:     {deep}  ← UNCHANGED! (true independent copy)")

print("\n💡 Use copy.deepcopy() for nested structures!")

---
## 4. Variable Scoping (LEGB Rule)

### The Rule
Python looks up variables in this order:
1. **L**ocal - inside current function
2. **E**nclosing - in outer function (for nested functions)
3. **G**lobal - at module level
4. **B**uilt-in - Python's built-in names

In [ ]:
# SCOPING EXAMPLE

x = "global"  # Global scope

def outer():
    x = "enclosing"  # Enclosing scope
    
    def inner():
        x = "local"  # Local scope
        print(f"  inner() sees: {x}")
    
    inner()
    print(f"  outer() sees: {x}")

outer()
print(f"  global sees: {x}")

In [ ]:
# ⚠️ THE TRAP: UnboundLocalError

x = 10  # Global

def broken_function():
    """
    THIS WILL FAIL!
    
    Python sees 'x = 20' and assumes x is LOCAL.
    But we try to print(x) before assigning it.
    """
    print(x)  # Error! x is local but not yet assigned
    x = 20    # This makes x local for the ENTIRE function

try:
    broken_function()
except UnboundLocalError as e:
    print(f"❌ UnboundLocalError: {e}")
    print("   Python decided x is local (because of x=20)")
    print("   But we tried to read it before assignment!")

In [ ]:
# ✅ SOLUTION: Use 'global' keyword

counter = 0

def increment():
    global counter  # Tell Python: use the global variable
    counter += 1
    print(f"  Counter is now: {counter}")

print(f"Initial counter: {counter}")
increment()
increment()
print(f"Final counter: {counter}")

print("\n💡 Better approach: Pass as argument and return new value")

---
## 5. Closures and Late Binding

### The Trap
```python
funcs = []
for i in range(3):
    funcs.append(lambda: i)  # ❌ All lambdas share same i!

[f() for f in funcs]  # [2, 2, 2] not [0, 1, 2]!
```

In [ ]:
# ❌ LATE BINDING TRAP

funcs = []
for i in range(3):
    # Lambda captures the VARIABLE i, not its VALUE
    # All lambdas point to the same 'i'
    funcs.append(lambda: i)

# After loop, i = 2
# All lambdas look up 'i' and find 2

print("❌ Late binding problem:")
print(f"   Expected: [0, 1, 2]")
print(f"   Got:      {[f() for f in funcs]}")
print(f"\n   All lambdas return {i} (final value of i)")

In [ ]:
# ✅ SOLUTION: Capture value with default argument

funcs = []
for i in range(3):
    # x=i captures the CURRENT value of i
    # Default argument is evaluated at function definition time
    funcs.append(lambda x=i: x)

print("✅ Capturing value with default argument:")
print(f"   Expected: [0, 1, 2]")
print(f"   Got:      {[f() for f in funcs]}")

# HOW IT WORKS:
# Iteration 0: lambda is created with x=0 as default
# Iteration 1: lambda is created with x=1 as default
# Iteration 2: lambda is created with x=2 as default

---
## 6. Truthiness and Falsy Values

### Falsy Values in Python
- `None`
- `False`
- `0`, `0.0`, `0j`
- `""` (empty string)
- `[]`, `()`, `{}`, `set()` (empty containers)

### Everything else is truthy!

In [ ]:
# FALSY VALUES

falsy_values = [
    (None, "None"),
    (False, "False"),
    (0, "0"),
    (0.0, "0.0"),
    ("", "empty string"),
    ([], "empty list"),
    ({}, "empty dict"),
    (set(), "empty set"),
]

print("Falsy values in Python:")
print("-" * 40)
for value, name in falsy_values:
    print(f"  bool({name:15}) = {bool(value)}")

In [ ]:
# ⚠️ THE TRAP: 0 is falsy!

count = 0  # Valid count of zero

# ❌ WRONG: This fails for count = 0
if count:
    print(f"Count is: {count}")
else:
    print("❌ count is falsy (but 0 is a valid value!)")

# ✅ CORRECT: Explicit check for None
if count is not None:
    print(f"✅ Count is: {count} (works correctly)")

print("\n💡 Use explicit 'is not None' when 0 is a valid value!")

In [ ]:
# IDIOMATIC EMPTY CHECKS

my_list = []

# ❌ Verbose (works but not Pythonic)
if len(my_list) == 0:
    print("❌ Verbose: len(my_list) == 0")

if my_list == []:
    print("❌ Comparing to empty list")

# ✅ Pythonic
if not my_list:
    print("✅ Pythonic: if not my_list")

# For non-empty check:
my_list = [1, 2, 3]
if my_list:
    print("✅ List has items")

---
## 7. Dictionary Key Gotchas

### Keys Must Be Hashable!
- ✅ Strings, numbers, tuples (of hashable items)
- ❌ Lists, dicts, sets (mutable = not hashable)

In [ ]:
# VALID DICTIONARY KEYS

d = {}
d["string"] = 1       # ✅ String
d[42] = 2             # ✅ Integer
d[3.14] = 3           # ✅ Float
d[(1, 2)] = 4         # ✅ Tuple (immutable)

print("Valid keys:")
for key, val in d.items():
    print(f"  {key!r}: {val}")

# ❌ INVALID: Lists are mutable
try:
    d[[1, 2]] = 5
except TypeError as e:
    print(f"\n❌ Cannot use list as key: {e}")

In [ ]:
# ⚠️ TRAP: 1 == 1.0 == True

d = {}
d[1] = "integer"
d[1.0] = "float"  # Overwrites! 1 == 1.0

print("Integer and float equal:")
print(f"  1 == 1.0: {1 == 1.0}")
print(f"  d = {d}")  # {1: 'float'}

# True == 1, False == 0
d2 = {}
d2[True] = "boolean true"
d2[1] = "integer one"  # Overwrites!

print(f"\nTrue and 1 equal:")
print(f"  True == 1: {True == 1}")
print(f"  d2 = {d2}")  # {True: 'integer one'}

---
## 8. Generator vs List Comprehension

### The Difference
- **List `[x for x in ...]`**: Creates all items NOW, stores in memory
- **Generator `(x for x in ...)`**: Creates items ON-DEMAND (lazy)

In [ ]:
import sys

# MEMORY COMPARISON

# List: All 10000 items in memory
list_data = [x**2 for x in range(10000)]
list_mem = sys.getsizeof(list_data)

# Generator: Items created one at a time
gen_data = (x**2 for x in range(10000))
gen_mem = sys.getsizeof(gen_data)

print("Memory usage for 10,000 squared numbers:")
print(f"  List:      {list_mem:,} bytes")
print(f"  Generator: {gen_mem:,} bytes")
print(f"  Savings:   {(1 - gen_mem/list_mem)*100:.1f}%")

In [ ]:
# WHEN TO USE WHICH?

# ✅ Use LIST when:
# - Need to iterate multiple times
# - Need len(), indexing, slicing
# - Data fits in memory

data = [1, 2, 3, 4, 5]
print(f"List length: {len(data)}")
print(f"First item: {data[0]}")
print(f"Slice: {data[1:3]}")

# ✅ Use GENERATOR when:
# - Large data that won't fit in memory
# - Only iterating once
# - Processing pipeline (chained operations)

gen = (x**2 for x in range(5))
print(f"\nGenerator type: {type(gen)}")
print(f"Generator items: {list(gen)}")
print(f"Exhausted - no more items: {list(gen)}")

---
## Summary: Quick Reference

| Trap | Correct Approach |
|------|------------------|
| `is` for values | Use `==` (except for None/True/False) |
| `def f(items=[])` | Use `items=None`, then `if items is None: items = []` |
| `b = a` (list) | Use `b = a.copy()` or `copy.deepcopy()` |
| `lambda: i` in loop | Use `lambda x=i: x` |
| `if value:` when 0 valid | Use `if value is not None:` |
| List for big data | Use generator `(x for x in ...)` |